# Compression Analysis: Longitudinal Modulus and Cooperative Diffusivity

## Geometry

**z** = compression direction (piston normal; gel spans $z_\text{gel,lo}$ to $z_\text{gel,hi}$) | Support fixed at base | Piston moves at constant velocity until $\varepsilon_{zz} = \Delta L / L_0$ reaches `comp_percent`, then freezes for stress relaxation.

## Theory

### Longitudinal Modulus — Two Methods

**Method 1 — Poroelastic stress decomposition (Voronoi):**
$$M_\text{Voronoi} = \frac{\sigma'_{zz}}{\varepsilon_{zz}}, \qquad \sigma'_{zz}(z) = \sigma_{p,zz}(z) + \sigma_{s,zz}(z) - p_p(z), \qquad p_p(z) = -\frac{\sigma_{s,zz}(z)}{\phi_s(z)}$$

Volume fractions $\phi_p$, $\phi_s$ are computed via Voronoi tessellation on the final relaxed frame. The CI on $M_\text{Voronoi}$ reflects spatial heterogeneity across gel bins.

**Method 2 — Piston force:**
$$M_\text{piston} = \frac{P_\text{piston}}{\varepsilon_{zz}}, \qquad P_\text{piston} = \frac{F_{z,\text{piston}}}{L_x L_y}$$

$F_{z,\text{piston}}$ is the total pairwise z-force on piston atoms (`compute reduce sum fz`, recorded via `fix print`). For a homogeneous gel at mechanical equilibrium the two estimates should agree; deviations signal spatial heterogeneity or incomplete relaxation.

### Cooperative Diffusivity — Displacement Formulation

Polymer volume fraction $\phi_p$ is proportional to polymer z-displacement $u_z$, so the cooperative diffusion PDE
$$\frac{\partial \phi_p}{\partial t} = D_c \frac{\partial^2 \phi_p}{\partial z^2}$$
is equivalent to
$$\frac{\partial u_z}{\partial t} = D_c \frac{\partial^2 u_z}{\partial z^2}$$

The displacement-based approach avoids per-frame Voronoi tessellation. $u_z(z,t)$ is recorded via `compute displace/atom` + `fix ave/chunk`, with reference positions set at Phase 2 start so $u_z(z,0) = 0$ by construction.

$D_c$ is extracted by fitting the even-mode Fourier cosine solution (wall symmetry eliminates odd modes):
$$u_z(z, t) = u_z^0 + (u_z^1 - u_z^0)\left[1 + 2\sum_{k=1}^{N} \cos\!\left(\frac{2\pi k\, z}{L_\text{gel}}\right) e^{-4\pi^2 k^2 \tau_\text{eff}}\right], \qquad \tau_\text{eff} = \frac{D_c\, t}{L_\text{gel}^2} + \frac{1}{16\pi}$$

where $u_z^0 = 0$ (support boundary; set by reference construction) and $u_z^1$ is the gel-mean displacement at the final snapshot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
import os
from pathlib import Path
import matplotlib.font_manager as fm

plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False
})

print('Imports successful')

## Configuration

In [ ]:
# ── CONFIG: only change these lines to switch datasets ─────────────────────
RUN_ID   = "rho04_p1.52_600k_2.5M"   # folder name inside flow_data_local/compression/
sim_name = "walled_slab_support_5beads_tall_rho04_p1.52_1.0_1.0_600000_1.0_1.0_2500000"
# ───────────────────────────────────────────────────────────────────────────

from pathlib import Path
DATA_DIR  = Path("../../flow_data_local/compression") / RUN_ID
PLOT_DIR  = Path("../../flow_data_local/plots/compression") / RUN_ID
TRAJ_FILE = Path("../../flow_data_local/traj_files.nosync") / f"gel_flow_{sim_name}.lammpstrj"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Analysis parameters ────────────────────────────────────────────────────
binWidth          = 2.0
phi_gel_threshold = 0.1
Ncount_min        = 200    # min atoms/bin for displacement gel-mask
ci_level          = 0.95
dt_lj             = 0.005  # LJ timestep
comp_percent      = 0.1
piston_area       = None   # sigma^2; auto-read from box_dimensions file;
                           # set manually here if that file is unavailable

## Helper Functions

In [ ]:
def read_print_file(filepath, col_names=None):
    """Read a LAMMPS fix print output (one row per timestep).
    Lines beginning with # are skipped.
    Returns a dict of column arrays keyed by col_names (or col_0, col_1, ...).
    """
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            rows.append([float(v) for v in line.split()])
    if not rows:
        raise ValueError(f'No data in {filepath}')
    arr = np.array(rows)
    if col_names is None:
        col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}


def read_ave_time_file(filepath):
    """Read a LAMMPS fix ave/time mode vector output.
    Returns list of (timestep, bin_indices, values_array).
    """
    data_by_time = []
    with open(filepath, 'r') as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            timestep, nrows = int(parts[0]), int(parts[1])
            values = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2:
                        values.append(float(vp[1]))
            if values:
                data_by_time.append((timestep, np.arange(1, len(values)+1), np.array(values)))
            i += nrows + 1
        else:
            i += 1
    return data_by_time


def read_ave_chunk_file(filepath):
    """Read a LAMMPS fix ave/chunk output.
    Returns list of (timestep, chunk_array) where chunk_array columns are:
    [0]=chunk_id  [1]=Coord1  [2]=Ncount  [3]=val1  ...
    Header line format: timestep nchunks [total-count]  (2 or 3 values).
    """
    snapshots = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) in (2, 3):
            try:
                timestep, nchunks = int(parts[0]), int(parts[1])
            except ValueError:
                i += 1
                continue
            rows = []
            for j in range(1, nchunks + 1):
                if i + j < len(lines):
                    rows.append([float(v) for v in lines[i + j].split()])
            if rows:
                snapshots.append((timestep, np.array(rows)))
            i += nchunks + 1
        else:
            i += 1
    return snapshots


def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    if not rows:
        raise ValueError(f'No data in {filepath}')
    if len(rows) == 2 and all(len(r)==1 for r in rows):
        return None, np.array([rows[0][0]]), np.array([rows[1][0]])
    if len(rows) == 1 and len(rows[0]) == 2:
        return None, np.array([rows[0][0]]), np.array([rows[0][1]])
    if all(len(r)==3 for r in rows):
        arr = np.array(rows)
        ts, col1, col2 = arr[:,0].astype(int).tolist(), arr[:,1], arr[:,2]
        ratio = np.nanmedian(col2) / np.nanmedian(col1)
        dL = col2 if ratio < 0.5 else col1 - col2
        return ts, col1, dL
    raise ValueError(f'Unrecognised format in {filepath}')


def read_lammpstrj_frame(filepath, frame_idx=0):
    with open(filepath) as f:
        lines = f.readlines()
    starts = [i for i,l in enumerate(lines) if 'ITEM: TIMESTEP' in l]
    if frame_idx < 0: frame_idx = len(starts) + frame_idx
    s = starts[frame_idx]
    e = starts[frame_idx+1] if frame_idx+1<len(starts) else len(lines)
    fl = lines[s:e]
    timestep = int(fl[1].strip())
    box = {'x': [float(v) for v in fl[5].split()],
           'y': [float(v) for v in fl[6].split()],
           'z': [float(v) for v in fl[7].split()]}
    atoms = [[int(p[0]),int(p[1]),int(p[2]),float(p[3]),float(p[4]),float(p[5])]
             for p in [l.split() for l in fl[9:9+int(fl[3].strip())]]]
    return timestep, box, atoms


def compute_volume_fractions_1d(atoms_data, box_bounds, bin_width, direction='z'):
    a = np.array(atoms_data)
    typ, pos = a[:,1].astype(int), a[:,3:6]
    pm, sm = (typ==1)|(typ==2), (typ==3)
    d = {'x':0,'y':1,'z':2}[direction]
    lo, hi = box_bounds[direction]
    edges = np.arange(lo, hi+bin_width, bin_width)
    centers = (edges[:-1]+edges[1:])/2
    phi_p, phi_s = np.zeros(len(centers)), np.zeros(len(centers))
    for i,(a_,b_) in enumerate(zip(edges[:-1],edges[1:])):
        mask = (pos[:,d]>=a_)&(pos[:,d]<b_)
        np_, ns_, nt = np.sum(pm&mask), np.sum(sm&mask), np.sum(mask)
        if nt>0: phi_p[i], phi_s[i] = np_/nt, ns_/nt
    return centers, phi_p, phi_s


def compute_volume_fractions_1d_voronoi(atoms_data, box_bounds, bin_width, direction='z'):
    import tess
    import time

    a = np.array(atoms_data)
    typ = a[:,1].astype(int)
    pos = a[:,3:6]

    xlo, xhi = box_bounds['x']
    ylo, yhi = box_bounds['y']
    zlo, zhi = box_bounds['z']
    L = np.array([xhi-xlo, yhi-ylo, zhi-zlo])
    origin = np.array([xlo, ylo, zlo])

    mobile_mask = (typ == 1) | (typ == 2) | (typ == 3)
    typ_m = typ[mobile_mask]
    pos_m = pos[mobile_mask]
    pos_wrapped = origin + (pos_m - origin) % L

    print(f'  Checking for duplicate positions...')
    _, unique_idx = np.unique(np.round(pos_wrapped, 6), axis=0, return_index=True)
    n_dupes = len(pos_wrapped) - len(unique_idx)
    if n_dupes > 0:
        print(f'  Warning: removing {n_dupes} duplicate positions')
        pos_wrapped = pos_wrapped[unique_idx]
        typ_m = typ_m[unique_idx]

    print(f'  Building Voronoi container ({len(pos_wrapped)} atoms)...')
    t0 = time.time()
    cntr = tess.Container(
        pos_wrapped,
        limits=((xlo, ylo, zlo), (xhi, yhi, zhi)),
        periodic=True
    )
    print(f'  Container built ({time.time()-t0:.1f}s)')

    print(f'  Extracting cell volumes...')
    t0 = time.time()
    voronoi_volumes = np.array([c.volume() for c in cntr])
    print(f'  Volumes extracted ({time.time()-t0:.1f}s)')

    box_vol = L[0] * L[1] * L[2]
    voro_total = np.sum(voronoi_volumes)
    if abs(voro_total - box_vol) / box_vol > 0.01:
        print(f'  Warning: Voronoi volume {voro_total:.2f} differs from box {box_vol:.2f} by '
              f'{100*abs(voro_total-box_vol)/box_vol:.1f}%')

    d = {'x': 0, 'y': 1, 'z': 2}[direction]
    lo, hi = box_bounds[direction]
    edges   = np.arange(lo, hi + bin_width, bin_width)
    centers = (edges[:-1] + edges[1:]) / 2

    pm = (typ_m == 1) | (typ_m == 2)
    sm = (typ_m == 3)

    phi_p = np.zeros(len(centers))
    phi_s = np.zeros(len(centers))

    for i, (a_, b_) in enumerate(zip(edges[:-1], edges[1:])):
        mask    = (pos_wrapped[:, d] >= a_) & (pos_wrapped[:, d] < b_)
        V_p     = np.sum(voronoi_volumes[mask & pm])
        V_s     = np.sum(voronoi_volumes[mask & sm])
        V_total = np.sum(voronoi_volumes[mask])
        if V_total > 0:
            phi_p[i] = V_p / V_total
            phi_s[i] = V_s / V_total

    return centers, phi_p, phi_s


def mean_ci(values, ci_level=0.95):
    v = values[~np.isnan(values)]
    n = len(v)
    if n==0: return np.nan, np.nan, np.nan
    if n==1: return v[0], v[0], v[0]
    m = np.mean(v)
    lo, hi = stats.t.interval(ci_level, df=n-1, loc=m, scale=stats.sem(v))
    return m, lo, hi


print('Helper functions defined')

## Step 1: Load All Data

In [ ]:
# ── Stress profiles ────────────────────────────────────────────────────────
stress_data = {}
for key, fp in [
    ('polymer_z', DATA_DIR / f'stress_z_polymer_{sim_name}.dat'),
    ('solvent_z', DATA_DIR / f'stress_z_solvent_{sim_name}.dat'),
    ('polymer_x', DATA_DIR / f'stress_x_polymer_{sim_name}.dat'),
    ('solvent_x', DATA_DIR / f'stress_x_solvent_{sim_name}.dat'),
    ('polymer_y', DATA_DIR / f'stress_y_polymer_{sim_name}.dat'),
    ('solvent_y', DATA_DIR / f'stress_y_solvent_{sim_name}.dat'),
]:
    print(f'  Reading {key} ...')
    stress_data[key] = read_ave_time_file(fp)

n_stress = len(stress_data['polymer_z'])
print(f'Stress snapshots: {n_stress}')

# Extract time series
all_timesteps  = [stress_data['polymer_z'][i][0] for i in range(n_stress)]
all_sigma_p_zz = [stress_data['polymer_z'][i][2] for i in range(n_stress)]
all_sigma_s_zz = [stress_data['solvent_z'][i][2] for i in range(n_stress)]
all_sigma_p_xx = [stress_data['polymer_x'][i][2] for i in range(n_stress)]
all_sigma_s_xx = [stress_data['solvent_x'][i][2] for i in range(n_stress)]
all_sigma_p_yy = [stress_data['polymer_y'][i][2] for i in range(n_stress)]
all_sigma_s_yy = [stress_data['solvent_y'][i][2] for i in range(n_stress)]

# Spatial coordinates from bin indices (bin k -> center at k*binWidth - binWidth/2)
bins_z   = stress_data['polymer_z'][0][1]
bins_x   = stress_data['polymer_x'][0][1]
bins_y   = stress_data['polymer_y'][0][1]
z_coords = bins_z * binWidth - binWidth / 2
x_coords = bins_x * binWidth - binWidth / 2
y_coords = bins_y * binWidth - binWidth / 2

# Normalised coordinates [0, 1] over each axis's bin span
z_norm = (z_coords - z_coords.min()) / (z_coords.max() - z_coords.min())
x_norm = (x_coords - x_coords.min()) / (x_coords.max() - x_coords.min())
y_norm = (y_coords - y_coords.min()) / (y_coords.max() - y_coords.min())

# ── Gel strain ────────────────────────────────────────────────────────────
ts_strain, L_arr, dL_arr = read_strain_file(DATA_DIR / f'strain_zz_{sim_name}.dat')
L_final   = float(L_arr[-1])
dL_final  = float(dL_arr[-1])
eps_final = dL_final / L_final
eps_arr   = dL_arr / L_arr
print(f'\nFinal: L={L_final:.4f}  ΔL={dL_final:.4f}  ε={eps_final:.4f}')

# ── Piston force ──────────────────────────────────────────────────────────
pf_data  = read_print_file(DATA_DIR / f'piston_force_{sim_name}.dat',
                            col_names=['step', 'F_piston_z'])
steps_pf = pf_data['step'].astype(int)
F_piston = pf_data['F_piston_z']
print(f'Piston force snapshots: {len(steps_pf)}  (t={steps_pf[0]} -> {steps_pf[-1]})')

# ── Box dimensions -> piston area ─────────────────────────────────────────
box_dims_file = DATA_DIR / f'box_dimensions_{sim_name}.dat'
if box_dims_file.exists():
    bd = read_print_file(box_dims_file, col_names=['step', 'lx', 'ly', 'lz'])
    piston_area = float(np.mean(bd['lx'])) * float(np.mean(bd['ly']))
    print(f'Piston area = {piston_area:.2f} sigma^2  '
          f'(lx={np.mean(bd["lx"]):.2f}, ly={np.mean(bd["ly"]):.2f})')
elif piston_area is None:
    raise ValueError(
        f'box_dimensions file not found at {box_dims_file} and piston_area '
        'not set in config. Set piston_area manually in the Config cell.')
else:
    print(f'Box dims file not found -- using config piston_area = {piston_area:.2f}')

# ── Displacement profiles (fix ave/chunk) ─────────────────────────────────
# Columns: [0]=chunk_id  [1]=Coord1(z, sigma)  [2]=Ncount  [3]=mean_uz
disp_snapshots = read_ave_chunk_file(DATA_DIR / f'disp_z_polymer_{sim_name}.dat')
n_disp  = len(disp_snapshots)
disp_ts = np.array([s[0] for s in disp_snapshots])
disp_z_raw  = disp_snapshots[0][1][:, 1]                          # (n_bins,)
disp_Ncount = np.array([s[1][:, 2] for s in disp_snapshots])      # (n_disp, n_bins)
disp_uz     = np.array([s[1][:, 3] for s in disp_snapshots])      # (n_disp, n_bins)
print(f'\nDisplacement snapshots: {n_disp}  (t={disp_ts[0]} -> {disp_ts[-1]})')

print(f'\nTimesteps: {all_timesteps[0]} -> {all_timesteps[-1]}')
print('Done.')

# ── True component z-binned stresses (σ_zz, σ_xx, σ_yy per species) ────────
# Written by slab_with_flow.lmp (fix avg_sigmazz/sigmaxx/sigmayy outputs).
# Used in the poroelastic decomposition:
#   σ'_comp(z) = σ_{p,comp}(z) + φ_p(z) · p_p(z),   p_p = σ_{s,comp}(z) / φ_s(z)
# Falls back to the isotropic z-profiles with a warning if files are absent
# (e.g. for simulation runs predating these outputs).
_comp_files = [
    ('polymer_zz',   DATA_DIR / f'sigmazz_polymer_{sim_name}.dat'),
    ('solvent_zz',   DATA_DIR / f'sigmazz_solvent_{sim_name}.dat'),
    ('polymer_xx_z', DATA_DIR / f'sigmaxx_polymer_{sim_name}.dat'),
    ('solvent_xx_z', DATA_DIR / f'sigmaxx_solvent_{sim_name}.dat'),
    ('polymer_yy_z', DATA_DIR / f'sigmayy_polymer_{sim_name}.dat'),
    ('solvent_yy_z', DATA_DIR / f'sigmayy_solvent_{sim_name}.dat'),
]
has_component_stresses = all(fp.exists() for _, fp in _comp_files)

if has_component_stresses:
    for key, fp in _comp_files:
        stress_data[key] = read_ave_time_file(fp)
    all_sigma_p_zz_comp = [stress_data['polymer_zz'][i][2]   for i in range(n_stress)]
    all_sigma_s_zz_comp = [stress_data['solvent_zz'][i][2]   for i in range(n_stress)]
    all_sigma_p_xx_comp = [stress_data['polymer_xx_z'][i][2] for i in range(n_stress)]
    all_sigma_s_xx_comp = [stress_data['solvent_xx_z'][i][2] for i in range(n_stress)]
    all_sigma_p_yy_comp = [stress_data['polymer_yy_z'][i][2] for i in range(n_stress)]
    all_sigma_s_yy_comp = [stress_data['solvent_yy_z'][i][2] for i in range(n_stress)]
    print('Component z-binned stresses loaded (σ_zz, σ_xx, σ_yy)')
else:
    # Placeholder: fall back to isotropic z-profiles
    all_sigma_p_zz_comp = all_sigma_p_zz
    all_sigma_s_zz_comp = all_sigma_s_zz
    all_sigma_p_xx_comp = all_sigma_p_zz   # wrong direction but avoids crash
    all_sigma_s_xx_comp = all_sigma_s_zz
    all_sigma_p_yy_comp = all_sigma_p_zz
    all_sigma_s_yy_comp = all_sigma_s_zz
    print('WARNING: component z-binned stress files not found — '
          'rerun slab_with_flow.lmp with σ_zz/σ_xx/σ_yy outputs enabled. '
          'Isotropic z-profiles used as placeholder; poroelastic results will be approximate.')


## Step 2: Partial Stress Evolution

Equilibration check: zz, xx, and yy partial stresses for polymer and solvent across all snapshots, colored early (dark) to late (bright). Convergence of $\sigma_{p,zz}$ and $\sigma_{s,zz}$ is the primary indicator of a relaxed final state.

In [ ]:
colors = plt.cm.viridis(np.linspace(0, 1, n_stress))
alpha  = 0.7

# ── Figure 1: zz components ───────────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
fig1.suptitle(
    f'Equilibration Check (zz): {sim_name}\n{n_stress} snapshots',
    fontsize=12, fontweight='bold')
ax_pzz, ax_szz = axes1
for i in range(n_stress):
    ax_pzz.plot(z_norm, all_sigma_p_zz[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_pzz.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_pzz.set(xlabel='$z/L_z$', ylabel=r'$\sigma_{p,zz}$',
           title=r'(a) Polymer $\sigma_{zz}$(z,t)', xlim=(0,1))
ax_pzz.grid(alpha=0.3)
for i in range(n_stress):
    ax_szz.plot(z_norm, all_sigma_s_zz[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_szz.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_szz.set(xlabel='$z/L_z$', ylabel=r'$\sigma_{s,zz}$',
           title=r'(b) Solvent $\sigma_{zz}$(z,t)', xlim=(0,1))
ax_szz.grid(alpha=0.3)
sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=Normalize(vmin=all_timesteps[0], vmax=all_timesteps[-1]))
sm.set_array([])
fig1.colorbar(sm, ax=[ax_pzz, ax_szz], fraction=0.046, pad=0.04).set_label('Timestep')
out1 = PLOT_DIR / f'equilibration_check_zz_{sim_name}.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
print(f'Saved: {out1}')
plt.show()

# ── Figure 2: xx and yy components ───────────────────────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
fig2.suptitle(
    f'Equilibration Check (xx/yy): {sim_name}\n{n_stress} snapshots',
    fontsize=12, fontweight='bold')
ax_pxx, ax_sxx = axes2[0]
ax_pyy, ax_syy = axes2[1]
for i in range(n_stress):
    ax_pxx.plot(x_norm, all_sigma_p_xx[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_pxx.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_pxx.set(xlabel='$x/L_x$', ylabel=r'$\sigma_{p,xx}$',
           title=r'(c) Polymer $\sigma_{xx}$(x,t)', xlim=(0,1))
ax_pxx.grid(alpha=0.3)
for i in range(n_stress):
    ax_sxx.plot(x_norm, all_sigma_s_xx[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_sxx.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_sxx.set(xlabel='$x/L_x$', ylabel=r'$\sigma_{s,xx}$',
           title=r'(d) Solvent $\sigma_{xx}$(x,t)', xlim=(0,1))
ax_sxx.grid(alpha=0.3)
for i in range(n_stress):
    ax_pyy.plot(y_norm, all_sigma_p_yy[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_pyy.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_pyy.set(xlabel='$y/L_y$', ylabel=r'$\sigma_{p,yy}$',
           title=r'(e) Polymer $\sigma_{yy}$(y,t)', xlim=(0,1))
ax_pyy.grid(alpha=0.3)
for i in range(n_stress):
    ax_syy.plot(y_norm, all_sigma_s_yy[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_syy.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_syy.set(xlabel='$y/L_y$', ylabel=r'$\sigma_{s,yy}$',
           title=r'(f) Solvent $\sigma_{yy}$(y,t)', xlim=(0,1))
ax_syy.grid(alpha=0.3)
sm2 = plt.cm.ScalarMappable(cmap='viridis',
                              norm=Normalize(vmin=all_timesteps[0], vmax=all_timesteps[-1]))
sm2.set_array([])
fig2.colorbar(sm2, ax=axes2, fraction=0.046, pad=0.04).set_label('Timestep')
out2 = PLOT_DIR / f'equilibration_check_xxyy_{sim_name}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved: {out2}')
plt.show()

## Step 3: Gel Strain and Piston Force History

$\varepsilon_{zz}(t)$ from the Rg-based gel thickness and $F_{z,\text{piston}}(t)$ from the pairwise contact force on piston atoms. The force rises during compression, plateaus near `comp_percent`, then decays during relaxation — the decay timescale is directly related to $D_c$.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
ax2 = ax1.twinx()

# ── Strain (left axis) ────────────────────────────────────────────────────
if ts_strain is not None:
    ax1.plot(ts_strain, eps_arr * 100, '-', color='steelblue', lw=2.5,
             label=r'$\varepsilon_{zz}$')
ax1.axhline(eps_final * 100, color='steelblue', ls=':', lw=1.5, alpha=0.55)
ax1.set_xlabel('Step')
ax1.set_ylabel(r'$\varepsilon_{zz}$ (%)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(bottom=0)

# ── Piston force (right axis) ─────────────────────────────────────────────
ax2.plot(steps_pf, F_piston, '-', color='firebrick', lw=2.0, alpha=0.85,
         label=r'$F_{z,\mathrm{piston}}$')
ax2.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
ax2.set_ylabel(r'$F_{z,\mathrm{piston}}$ (LJ)', color='firebrick')
ax2.tick_params(axis='y', labelcolor='firebrick')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=16, loc='center right')

ax1.set_title(f'Gel Strain and Piston Force History  |  {sim_name}')
ax1.grid(alpha=0.3)

first_disp_ts = int(disp_ts[0])
idx_strain = np.argmin(np.abs(np.array(ts_strain, dtype=int) - first_disp_ts))
eps_at_start = float(eps_arr[idx_strain])
ax1.axvline(first_disp_ts, color='k', ls='--', lw=1.5, alpha=0.6,
            label=f'Displacement start (t={first_disp_ts})')

out_hist = PLOT_DIR / f'strain_force_history_{sim_name}.png'
plt.savefig(out_hist, dpi=150, bbox_inches='tight')
print(f'Saved: {out_hist}')
plt.show()


print(f'First displacement snapshot:  t = {first_disp_ts}')
print(f'Gel strain at that step:       ε = {eps_at_start:.4f}  ({eps_at_start*100:.2f}%)')
print(f'Target strain (comp_percent):  ε = {comp_percent}')
print(f'Final strain:  ε_zz = {eps_final:.4f}  ({eps_final*100:.2f}%)')
print(f'Gel thickness: L_0  = {L_final:.2f} sigma   ΔL = {dL_final:.2f} sigma')
print(f'Piston area = {piston_area:.2f} sigma^2  ')
print(f'Final piston force (step {steps_pf[-1]}): F_z = {F_piston[-1]:.4f} LJ')

## Step 4: Displacement Profile Evolution $u_z(z,t)$

Per-bin mean z-displacement from `fix ave/chunk`, colored early (dark) to late (bright). Edge bins with fewer than `Ncount_min` atoms are excluded. Reference is Phase 2 start so $u_z(z,0) = 0$; negative values indicate downward compression.

In [ ]:
Ncount_mean_all = np.mean(disp_Ncount, axis=0)
gel_mask_disp   = Ncount_mean_all > Ncount_min
z_gel_disp      = disp_z_raw[gel_mask_disp]

fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
norm_t  = Normalize(vmin=disp_ts[0], vmax=disp_ts[-1])
cmap    = plt.cm.viridis

for ts, snap in disp_snapshots:
    uz_i = snap[:, 3]
    ax.plot(z_gel_disp, uz_i[gel_mask_disp],
            '-', color=cmap(norm_t(ts)), lw=1.2, alpha=0.7)

ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
ax.set(xlabel=r'$z$ ($\sigma$)', ylabel=r'$u_z$ ($\sigma$)',
       title=r'Polymer displacement $u_z(z,\,t)$  --  reference: Phase 2 start')
ax.grid(alpha=0.3)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_t)
sm.set_array([])
fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02).set_label('Timestep')

out_uz = PLOT_DIR / f'uz_evolution_{sim_name}.png'
plt.savefig(out_uz, dpi=150, bbox_inches='tight')
print(f'Saved: {out_uz}')
plt.show()

## Step 5: Cooperative Diffusivity $D_c$ — Fourier Sine Fit

Fit $D_c$ to the even-mode sine series (Dirichlet BCs: zero displacement at both walls; zero IC):

$$u_z(z, t) = \sum_{k=1}^{N} A_k \left[1 - e^{-4\pi^2 k^2 \tau}\right] \sin\!\left(\frac{2\pi k\, z}{L_\text{gel}}\right), \qquad \tau = \frac{D_c\, t}{L_\text{gel}^2}$$

**Physical basis:**
- BCs: $u_z = 0$ at $\hat{z} = 0$ and $\hat{z} = 1$ — polymer cannot cross the walls (Dirichlet, not Neumann). No-flux on $\phi$ ($\partial\phi/\partial z = 0$ at walls) implies $\partial^2 u_z/\partial z^2 = 0$ at walls via $\phi - \bar\phi \approx -\bar\phi\,\partial u_z/\partial z$; together with the PDE this forces $\partial u_z/\partial t = 0$ at the walls, so the Dirichlet condition is self-consistently maintained.
- IC: $u_z = 0$ everywhere at $t = 0$ — displacement is measured from Phase 2 start. The $[1 - e^{-\lambda_k t}]$ factor satisfies this exactly without any virtual-time offset.
- Even modes $n = 2k$ (i.e.\ $\sin(2\pi k\hat{z})$) enforce antisymmetry about the gel midpoint ($u_z(\hat{z}=\tfrac{1}{2}) = 0$), consistent with the observed symmetric initial $\phi$ profile.

For fixed $D_c$, the amplitudes $\{A_k\}$ are solved exactly by linear least squares. $D_c$ is then found by scalar minimisation of the total squared residual over the first `frac_early` fraction of snapshots.

In [ ]:
# ── USER INPUTS ────────────────────────────────────────────────────────────
edge_margin  = 0       # extra gel-edge bins to drop before fitting
binWidth_Dc  = binWidth
frac_early   = 0.4     # use first frac_early fraction of snapshots
N_modes      = 1       # Fourier modes k = 1..N
Dc_bounds    = (1e-6, 1.0)
trim_bins    = 2       # interior trim for reference-value extraction
# ── END USER INPUTS ────────────────────────────────────────────────────────

# ── Match displacement snapshots to stress timesteps ──────────────────────
# Displacement and stress share nfreq so timesteps align by construction.
disp_ts_to_idx = {int(ts): i for i, ts in enumerate(disp_ts)}
uz_matched = []
Nc_matched = []
for ts in all_timesteps:
    if ts in disp_ts_to_idx:
        idx = disp_ts_to_idx[ts]
    else:
        nearest = int(min(disp_ts, key=lambda t: abs(t - ts)))
        print(f'  Warning: stress ts={ts} not in displacement, using nearest={nearest}')
        idx = disp_ts_to_idx[nearest]
    uz_matched.append(disp_uz[idx])
    Nc_matched.append(disp_Ncount[idx])
uz_matched = np.array(uz_matched)   # (n_stress, n_disp_bins)
Nc_matched = np.array(Nc_matched)

# ── Gel domain ────────────────────────────────────────────────────────────
gel_bins_d = np.where(Nc_matched[0] > Ncount_min)[0]
if len(gel_bins_d) == 0:
    raise RuntimeError(f'No gel bins with Ncount > {Ncount_min} in first snapshot')

i_left_d  = gel_bins_d[0]  + edge_margin
i_right_d = gel_bins_d[-1] - edge_margin + 1

z_gel_d     = disp_z_raw[i_left_d:i_right_d]
uz_gel      = uz_matched[:, i_left_d:i_right_d]   # (n_stress, n_gel_bins)
L_gel_sigma = (i_right_d - i_left_d) * binWidth_Dc
zeta_sigma  = z_gel_d - z_gel_d[0]
zhat        = zeta_sigma / L_gel_sigma

# ── Time arrays ───────────────────────────────────────────────────────────
times     = np.array(all_timesteps, dtype=float)
dt_lj_arr = (times - times[0]) * dt_lj   # elapsed time in LJ tau

# ── Early-snapshot selection ──────────────────────────────────────────────
max_dt_steps = (times[-1] - times[0]) * frac_early
early_idx    = np.where((dt_lj_arr <= max_dt_steps) & (dt_lj_arr > 0))[0]

if len(early_idx) == 0:
    raise RuntimeError(
        f'No early snapshots with frac_early={frac_early}. '
        f'Total steps={times[-1]-times[0]:.0f}. Try increasing frac_early.')

print(f'L_gel = {L_gel_sigma:.1f} sigma  |  {len(zhat)} bins  |  '
      f'{len(early_idx)} early snapshots  '
      f'(first {frac_early*100:.0f}% = {max_dt_steps*dt_lj:.1f} tau)')



In [ ]:
# ── Trim mask ─────────────────────────────────────────────────────────────
fit_mask        = np.ones(len(zhat), dtype=bool)
fit_mask[:trim_bins]  = False
fit_mask[-trim_bins:] = False
zhat_fit = zhat[fit_mask]
print(f'{np.sum(fit_mask)} bins in fit domain  (trim_bins={trim_bins} each side)')

# ── Fourier sine model ─────────────────────────────────────────────────────
# u_z(z,t) = sum_{k=1}^{N} A_k * [1 - exp(-4*pi^2*k^2 * tau)] * sin(2*pi*k*zh)
# where  tau = Dc * t / L_gel^2
# BCs: u_z = 0 at zh = 0 and zh = 1  (Dirichlet)
# IC:  u_z = 0 at t = 0              (reference: Phase 2 start)
# Even-n modes sin(2*pi*k*zh) are antisymmetric about zh = 0.5.

def build_basis(zh, Dc, t_lj_val):
    """Design matrix (n_pts, N_modes): column k is [1-exp(-4*pi^2*k^2*tau)]*sin(2*pi*k*zh)."""
    tau  = Dc * t_lj_val / L_gel_sigma**2
    cols = []
    for k in range(1, N_modes + 1):
        decay = 1.0 - np.exp(-4.0 * np.pi**2 * k**2 * tau)
        cols.append(decay * np.sin(2.0 * np.pi * k * zh))
    return np.column_stack(cols)   # (n_pts, N_modes)

def fit_amplitudes(Dc):
    """Linear least-squares solve for A_k given Dc (stacked over early snapshots)."""
    rows_y, rows_X = [], []
    for i in early_idx:
        rows_y.append(uz_gel[i][fit_mask])
        rows_X.append(build_basis(zhat_fit, Dc, dt_lj_arr[i]))
    A, _, _, _ = np.linalg.lstsq(
        np.vstack(rows_X), np.concatenate(rows_y), rcond=None)
    return A

def residual(Dc):
    A  = fit_amplitudes(Dc)
    ss = 0.0
    for i in early_idx:
        pred = build_basis(zhat_fit, Dc, dt_lj_arr[i]) @ A
        ss  += np.sum((pred - uz_gel[i][fit_mask])**2)
    return ss

def fourier_model(zh, Dc, A_k, t_lj_val):
    return build_basis(zh, Dc, t_lj_val) @ A_k

# ── Fit ───────────────────────────────────────────────────────────────────
res      = minimize_scalar(residual, bounds=Dc_bounds, method='bounded')
Dc_fit   = res.x
A_k_fit  = fit_amplitudes(Dc_fit)

print(f'D_c = {Dc_fit:.4e} sigma^2/tau  (N_modes={N_modes})')
for k, A in enumerate(A_k_fit, 1):
    print(f'  A_{k} = {A:.4f} sigma  (amplitude of sin(2*pi*{k}*zh))')

# ── R^2 ───────────────────────────────────────────────────────────────────
R2_per = []
for i in early_idx:
    y    = uz_gel[i][fit_mask]
    yhat = fourier_model(zhat_fit, Dc_fit, A_k_fit, dt_lj_arr[i])
    ss_r = np.sum((y - yhat)**2)
    ss_t = np.sum((y - np.mean(y))**2)
    R2_per.append(1.0 - ss_r / ss_t if ss_t > 1e-30 else np.nan)

uz_data_all = np.concatenate([uz_gel[i][fit_mask] for i in early_idx])
uz_pred_all = np.concatenate([fourier_model(zhat_fit, Dc_fit, A_k_fit, dt_lj_arr[i])
                               for i in early_idx])
ss_res = np.sum((uz_data_all - uz_pred_all)**2)
ss_tot = np.sum((uz_data_all - np.mean(uz_data_all))**2)
R2     = 1.0 - ss_res / ss_tot

print(f'R^2 = {R2:.6f}  (aggregate over early snapshots)')
print('R^2 per snapshot:')
for idx, r2 in zip(early_idx, R2_per):
    print(f'  step {times[idx]:.0f}  ->  R^2 = {r2:.3f}')

# ── Asymptotic profile (t -> inf) ─────────────────────────────────────────
zhat_fine = np.linspace(0, 1, 600)
uz_inf    = sum(A_k_fit[k - 1] * np.sin(2.0 * np.pi * k * zhat_fine)
                for k in range(1, N_modes + 1))
print(f'\nAsymptotic u_z extrema: min={uz_inf.min():.4f}, max={uz_inf.max():.4f} sigma')

# ── Sequential-difference plot  Δu_z(z, t_i) = u_z(t_i) − u_z(t_{i-1}) ──
fig_d, ax_d = plt.subplots(figsize=(10, 5), constrained_layout=True)
norm_t_d    = Normalize(vmin=times[early_idx[1]], vmax=times[early_idx[-1]])

for j in range(1, len(early_idx)):
    i_curr = early_idx[j]
    i_prev = early_idx[j - 1]
    delta  = uz_gel[i_curr][fit_mask] - uz_gel[i_prev][fit_mask]
    ax_d.plot(zhat_fit, delta, 'o-', color=cmap_fit(norm_t_d(times[i_curr])),
              ms=3, lw=1.2, alpha=0.7)

ax_d.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_d.set(xlabel=r'$\hat{z} = z / L_\mathrm{gel}$',
         ylabel=r'$u_z(t_i) - u_z(t_{i-1})$  ($\sigma$)',
         xlim=(0, 1))
ax_d.grid(alpha=0.3)
ax_d.set_title(r'Sequential displacement increments $\Delta u_z = u_z(t_i) - u_z(t_{i-1})$')

sm_d = plt.cm.ScalarMappable(cmap=cmap_fit, norm=norm_t_d)
sm_d.set_array([])
fig_d.colorbar(sm_d, ax=ax_d, fraction=0.025, pad=0.02).set_label('Timestep $t_i$')

out_delta = PLOT_DIR / f'Dc_uz_increments_{sim_name}.png'
plt.savefig(out_delta, dpi=150, bbox_inches='tight')
print(f'Saved: {out_delta}')
plt.show()

# ── Plot ──────────────────────────────────────────────────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
norm_t   = Normalize(vmin=times[early_idx[0]], vmax=times[early_idx[-1]])
cmap_fit = plt.cm.viridis

for i in early_idx:
    c = cmap_fit(norm_t(times[i]))
    ax_l.plot(zhat, uz_gel[i], 'o-', color=c, ms=3, alpha=0.6)
    ax_r.plot(zhat, uz_gel[i], 'o-', color=c, ms=3, alpha=0.35)
    ax_r.plot(zhat_fine, fourier_model(zhat_fine, Dc_fit, A_k_fit, dt_lj_arr[i]),
              '-', color=c, lw=2.0)

ax_r.plot(zhat_fine, uz_inf, 'k--', lw=1.8, label=r'$u_z(t\to\infty)$')

for ax in (ax_l, ax_r):
    ax.axhline(0, color='steelblue', ls=':', lw=1.5, label=r'$u_z = 0$')
    ax.set(xlabel=r'$\hat{z} = z / L_\mathrm{gel}$',
           ylabel=r'$u_z$ ($\sigma$)', xlim=(0, 1))
    ax.grid(alpha=0.3)

ax_l.legend(fontsize=14)
ax_l.set_title(r'Raw $u_z(z,\,t)$ — early snapshots')
ax_r.legend(fontsize=12)
ax_r.set_title(rf'Sine fit ($N={N_modes}$):  $D_c = {Dc_fit:.2e}\ \sigma^2/\tau$,  $R^2 = {R2:.3f}$')

sm = plt.cm.ScalarMappable(cmap=cmap_fit, norm=norm_t)
sm.set_array([])
fig.colorbar(sm, ax=[ax_l, ax_r], fraction=0.015, pad=0.04).set_label('Timestep')
fig.suptitle(f'Cooperative diffusivity fit  |  {sim_name}', fontsize=12, fontweight='bold')

out_fourier = PLOT_DIR / f'Dc_fourier_fit_{sim_name}.png'
plt.savefig(out_fourier, dpi=150, bbox_inches='tight')
print(f'\nSaved: {out_fourier}')
plt.show()


## Step 6: Final-State Profiles

Voronoi volume fractions computed once for the final trajectory frame. Used for the poroelastic decomposition: pore pressure $p_p(z)$, effective network stress $\sigma'_{zz}(z)$, and longitudinal modulus $M(z) = \sigma'_{zz}(z)/\varepsilon_{zz}$.

In [ ]:
from scipy.interpolate import interp1d

idx_final = -1  # last stress snapshot

# Use true component z-binned stresses (σ_zz, σ_xx, σ_yy per species).
# 'has_component_stresses' and the _comp arrays are set in the Load cell above.
sig_p_final    = np.array(all_sigma_p_zz_comp[idx_final])  # σ_{p,zz}(z)
sig_s_final    = np.array(all_sigma_s_zz_comp[idx_final])  # σ_{s,zz}(z)
sig_p_final_xx = np.array(all_sigma_p_xx_comp[idx_final])  # σ_{p,xx}(z)
sig_s_final_xx = np.array(all_sigma_s_xx_comp[idx_final])  # σ_{s,xx}(z)
sig_p_final_yy = np.array(all_sigma_p_yy_comp[idx_final])  # σ_{p,yy}(z)
sig_s_final_yy = np.array(all_sigma_s_yy_comp[idx_final])  # σ_{s,yy}(z)

if not has_component_stresses:
    print('NOTE: using isotropic placeholder — component stress values are approximate.')

# ── Voronoi volume fractions — final frame only ───────────────────────────
print('Computing Voronoi volume fractions for final frame...')
_, box_final, atoms_final = read_lammpstrj_frame(TRAJ_FILE, -1)
box_bounds_final = {'x': box_final['x'], 'y': box_final['y'], 'z': box_final['z']}

z_vor, phi_p_vor, phi_s_vor = compute_volume_fractions_1d_voronoi(
    atoms_final, box_bounds_final, binWidth, 'z')

zlo_f, zhi_f = box_final['z']
Lz_f         = zhi_f - zlo_f
z_vor_norm   = (z_vor - zlo_f) / Lz_f

# Interpolate Voronoi phi onto the stress bin z-grid (shared for all components)
phi_p_final = interp1d(z_vor, phi_p_vor, bounds_error=False, fill_value=0.0)(z_coords)
phi_s_final = interp1d(z_vor, phi_s_vor, bounds_error=False, fill_value=0.0)(z_coords)

phi_s_final_safe = np.where(phi_s_final > 1e-6, phi_s_final, np.nan)

In [ ]:
# ── Poroelastic decomposition: σ'_comp(z) = σ_{p,comp}(z) + φ_p(z)·p_p(z) ─
# p_p(z) = σ_{s,comp}(z) / φ_s(z)   (pore pressure from solvent component stress)
# Applied identically for zz, xx, yy — all quantities are z-binned profiles.

# zz ──────────────────────────────────────────────────────────────────────
p_p_final        = 1/3*(sig_s_final + sig_s_final_xx + sig_s_final_yy)/phi_s_final_safe
sig_prime_final  = sig_p_final + sig_s_final - np.where(np.isnan(p_p_final), 0.0, p_p_final)

if abs(eps_final) < 1e-12:
    raise ValueError('Final strain ~= 0 -- cannot compute M')
M_final = sig_prime_final / eps_final

# xx ──────────────────────────────────────────────────────────────────────
sig_prime_final_xx = sig_p_final_xx + sig_s_final_xx - np.where(np.isnan(p_p_final), 0.0, p_p_final)

# yy ──────────────────────────────────────────────────────────────────────
sig_prime_final_yy = sig_p_final_yy + sig_s_final_yy - np.where(np.isnan(p_p_final), 0.0, p_p_final)

# ── Summary ───────────────────────────────────────────────────────────────
interior_mask = (phi_p_final > 0.05) & (phi_s_final > 0.05)
phi_p_mean    = float(np.nanmean(phi_p_final[interior_mask]))
phi_s_mean    = float(np.nanmean(phi_s_final[interior_mask]))

print("sig_p_zz interior mean:", np.nanmean(sig_p_final[interior_mask]))
print("sig_s_zz interior mean:", np.nanmean(sig_s_final[interior_mask]))
print("p_p interior mean:    ", np.nanmean(p_p_final[interior_mask]))
print(f'phi_p mean (interior): {phi_p_mean:.4f}')
print(f'phi_s mean (interior): {phi_s_mean:.4f}')
print(f'p_pore_zz mean (interior): {float(np.nanmean(np.where(np.isnan(p_p_final), np.nan, p_p_final)[interior_mask])):.4f}')
print(f"sigma'_zz mean (interior): {float(np.nanmean(sig_prime_final[interior_mask])):.4f}")
print(f"sigma'_xx mean (interior): {float(np.nanmean(sig_prime_final_xx[interior_mask])):.4f}")
print(f"sigma'_yy mean (interior): {float(np.nanmean(sig_prime_final_yy[interior_mask])):.4f}")

In [ ]:
# ── Count-based phi (comparison only) ─────────────────────────────────────
z_cnt, phi_p_cnt, phi_s_cnt = compute_volume_fractions_1d(
    atoms_final, box_bounds_final, binWidth, 'z')

# ── Plot A: count-based vs Voronoi ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
fig.suptitle('Count-based vs Voronoi Volume Fractions (final frame)', fontweight='bold')
for ax, (phi_p, phi_s, z, label) in zip(axes, [
    (phi_p_cnt, phi_s_cnt, z_cnt, 'Count-based'),
    (phi_p_vor, phi_s_vor, z_vor, 'Voronoi')
]):
    ax.plot((z - zlo_f)/Lz_f, phi_p, 'o-', color='steelblue', ms=3, label=r'$\phi_p$')
    ax.plot((z - zlo_f)/Lz_f, phi_s, 's--', color='coral', ms=3, label=r'$\phi_s$')
    ax.legend()
    ax.set(xlabel='$z/L_z$', ylabel='Volume Fraction', title=label, xlim=(0,1))
    ax.grid(alpha=0.3)
plt.savefig(PLOT_DIR / f'volfrac_comparison_{sim_name}.png', dpi=150)
plt.show()

# ── Plot B: 2x2 final-state profiles ─────────────────────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
fig2.suptitle(
    f'Final Equilibrated State (Voronoi phi): {RUN_ID}\nt = {all_timesteps[idx_final]}',
    fontsize=12, fontweight='bold')

ax = axes2[0, 0]
ax.plot(z_vor_norm, phi_p_vor, 'o-', label=r'$\phi_p$', color='steelblue', ms=3)
ax.plot(z_vor_norm, phi_s_vor, 's--', label=r'$\phi_s$', color='coral', ms=3)
ax.legend()
ax.set(xlabel='$z/L_z$', ylabel='$\phi(z)$', title='(a) Volume fraction', xlim=(0,1))
ax.grid(alpha=0.3)

ax = axes2[0, 1]
ax.plot(z_norm, p_p_final, 'o-', color='green', ms=3)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='$z/L_z$', ylabel=r'$p_p$', title='(b) Pore pressure $p_p(z)$', xlim=(0,1))
ax.grid(alpha=0.3)

ax = axes2[1, 0]
ax.plot(z_norm, sig_prime_final, 'o-', color='purple', ms=3)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='$z/L_z$', ylabel=r"$\sigma'_{zz}$",
       title=r"(c) Network stress $\sigma'_{zz}(z)$", xlim=(0,1))
ax.grid(alpha=0.3)

ax = axes2[1, 1]
ax.plot(z_norm, M_final, 'o-', color='crimson', ms=3)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='$z/L_z$', ylabel='$M$',
       title=r'(d) Longitudinal modulus $M = \sigma^{\prime}_{zz}/\varepsilon$', xlim=(0,1))
ax.grid(alpha=0.3)

out2 = PLOT_DIR / f'final_state_profiles_{sim_name}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved: {out2}')
plt.show()

# ── Plot C: xx, yy, and zz network stress profiles (all z-binned) ─────────
# All three components are now z-binned profiles: σ'_comp(z) = σ_{p,comp} + φ_p·p_p
sig_prime_xx_mean = float(np.nanmean(sig_prime_final_xx[interior_mask]))
sig_prime_yy_mean = float(np.nanmean(sig_prime_final_yy[interior_mask]))
sig_prime_zz_mean = float(np.nanmean(sig_prime_final[interior_mask]))
print(f"sigma'_xx mean (interior): {sig_prime_xx_mean:.4f}")
print(f"sigma'_yy mean (interior): {sig_prime_yy_mean:.4f}")
print(f"sigma'_zz mean (interior): {sig_prime_zz_mean:.4f}")

fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fig3.suptitle(
    f'Final-State Network Stress Components (z-binned): {RUN_ID}\nt = {all_timesteps[idx_final]}',
    fontsize=12, fontweight='bold')

ax_xx, ax_yy, ax_zz = axes3

ax_xx.plot(z_norm, sig_prime_final_xx, 'o-', color='steelblue', ms=3,
           label=f'mean = {sig_prime_xx_mean:.4g}')
ax_xx.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_xx.set(xlabel='$z/L_z$', ylabel=r"$\sigma'_{xx}$",
          title=r"(a) Network stress $\sigma'_{xx}(z)$", xlim=(0, 1))
ax_xx.grid(alpha=0.3)
ax_xx.legend(fontsize=12)

ax_yy.plot(z_norm, sig_prime_final_yy, 'o-', color='coral', ms=3,
           label=f'mean = {sig_prime_yy_mean:.4g}')
ax_yy.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_yy.set(xlabel='$z/L_z$', ylabel=r"$\sigma'_{yy}$",
          title=r"(b) Network stress $\sigma'_{yy}(z)$", xlim=(0, 1))
ax_yy.grid(alpha=0.3)
ax_yy.legend(fontsize=12)

ax_zz.plot(z_norm, sig_prime_final, 'o-', color='purple', ms=3,
           label=f'mean = {sig_prime_zz_mean:.4g}')
ax_zz.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_zz.set(xlabel='$z/L_z$', ylabel=r"$\sigma'_{zz}$",
          title=r"(c) Network stress $\sigma'_{zz}(z)$", xlim=(0, 1))
ax_zz.grid(alpha=0.3)
ax_zz.legend(fontsize=12)

out3 = PLOT_DIR / f'final_state_network_stress_components_{sim_name}.png'
plt.savefig(out3, dpi=150, bbox_inches='tight')
print(f'Saved: {out3}')
plt.show()

## Step 7: Longitudinal Modulus — Voronoi vs Piston Force

Two independent estimates of $M$:

- **Voronoi**: spatially resolved $M(z) = \sigma'_{zz}(z)/\varepsilon_{zz}$, gel-averaged with 95% CI from bin-to-bin spatial scatter.
- **Piston force**: $M_\text{piston} = P_\text{piston}/\varepsilon_{zz}$ where $P_\text{piston} = F_z / A$; single scalar from `fix print`.

Agreement validates the Voronoi decomposition. Deviations can indicate incomplete relaxation (piston still loading dynamically), non-affine deformation, or Voronoi tessellation errors.

In [ ]:
print('=' * 70)
print('LONGITUDINAL MODULUS: VORONOI vs PISTON FORCE')
print('=' * 70)

# ── Method 1: Voronoi-based M ─────────────────────────────────────────────
gel_mask   = phi_p_final > phi_gel_threshold
M_gel      = M_final[gel_mask]
M_gel      = M_gel[~np.isnan(M_gel)]
n_gel_bins = len(M_gel)

if n_gel_bins == 0:
    raise ValueError('No valid M values in gel region')

M_mean, M_ci_lo, M_ci_hi = mean_ci(M_gel, ci_level)
ci_pct = int(ci_level * 100)

print(f'\nMethod 1 -- Voronoi decomposition:')
print(f'  Gel bins: {n_gel_bins}  (phi_p > {phi_gel_threshold})')
print(f'  M_gel range: [{M_gel.min():.4f}, {M_gel.max():.4f}]')
if n_gel_bins > 1:
    s = np.std(M_gel, ddof=1)
    sem = stats.sem(M_gel)
    df = n_gel_bins - 1
    t_crit = stats.t.ppf((1 + ci_level) / 2, df)
    print(f'  s = {s:.6f}   SEM = {sem:.6f}   t_crit(df={df}) = {t_crit:.4f}')
print(f'  M_voronoi = {M_mean:.4f}  [{M_ci_lo:.4f}, {M_ci_hi:.4f}]  ({ci_pct}% CI)')

# ── Method 2: Piston force ────────────────────────────────────────────────
# Use the piston force value nearest to the final stress snapshot timestep.
ts_final       = all_timesteps[idx_final]
idx_pf_final   = int(np.argmin(np.abs(steps_pf - ts_final)))
F_piston_final = float(F_piston[idx_pf_final])
P_piston_final = F_piston_final / piston_area
M_piston       = P_piston_final / eps_final

print(f'\nMethod 2 -- Piston force:')
print(f'  Nearest piston-force step: {steps_pf[idx_pf_final]}  '
      f'(stress snapshot: {ts_final})')
print(f'  F_z = {F_piston_final:.4f} LJ  |  '
      f'A = {piston_area:.2f} sigma^2  |  '
      f'P = F/A = {P_piston_final:.4f}')
print(f'  M_piston = P / eps_zz = {M_piston:.4f}')

print(f'\nComparison:')
print(f'  M_voronoi = {M_mean:.4f}  [{M_ci_lo:.4f}, {M_ci_hi:.4f}]')
print(f'  M_piston  = {M_piston:.4f}')
print(f'  Ratio M_piston / M_voronoi = {M_piston / M_mean:.4f}')
print(f'\n  eps_zz = {eps_final:.4f}  |  L = {L_final:.2f}  |  dL = {dL_final:.2f}')
print('=' * 70)

# ── Comparison plot ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

ax.errorbar([0], [M_mean],
            yerr=[[M_mean - M_ci_lo], [M_ci_hi - M_mean]],
            fmt='o', ms=12, color='steelblue', capsize=8, lw=2.5,
            label=f'Voronoi  $M = {M_mean:.3f}$\n{ci_pct}% CI: [{M_ci_lo:.3f}, {M_ci_hi:.3f}]')
ax.plot([1], [M_piston], 's', ms=12, color='firebrick',
        label=f'Piston  $M = {M_piston:.3f}$')

ax.axhline(M_mean,   color='steelblue', ls='--', lw=1.2, alpha=0.5)
ax.axhline(M_piston, color='firebrick', ls='--', lw=1.2, alpha=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Voronoi', 'Piston force'], fontsize=20)
ax.set_ylabel('$M$ (LJ units)')
ax.set_title(f'Longitudinal Modulus Comparison\n'
             f'{RUN_ID}  |  $\\varepsilon_{{zz}} = {eps_final:.3f}$')
ax.legend(fontsize=15, loc='upper right')
ax.set_xlim(-0.5, 1.5)
ax.grid(axis='y', alpha=0.3)

out_M = PLOT_DIR / f'M_comparison_{sim_name}.png'
plt.savefig(out_M, dpi=150, bbox_inches='tight')
print(f'Saved: {out_M}')
plt.show()

## Optional: Save Data

In [ ]:
out_data = PLOT_DIR / f'compression_modulus_final_{sim_name}.npz'
np.savez(out_data,
         # Coordinates
         z_coords=z_coords, z_norm=z_norm,
         z_vor=z_vor, z_vor_norm=z_vor_norm,
         # Timestep
         final_timestep=all_timesteps[idx_final],
         # Strain
         L=L_final, dL=dL_final, eps=eps_final,
         # Volume fractions (Voronoi)
         phi_p=phi_p_final, phi_s=phi_s_final,
         phi_p_vor=phi_p_vor, phi_s_vor=phi_s_vor,
         # Stress decomposition
         p_p=p_p_final, sigma_p_zz=sig_p_final, sigma_s_zz=sig_s_final,
         sigma_prime_zz=sig_prime_final,
         # Longitudinal modulus
         M=M_final,
         M_mean=M_mean, M_ci_lo=M_ci_lo, M_ci_hi=M_ci_hi,
         M_piston=M_piston, piston_area=piston_area,
         # Cooperative diffusivity
         Dc_fit=Dc_fit,
         # Metadata
         ci_level=ci_level, n_gel_bins=n_gel_bins,
         run_id=RUN_ID, sim_name=sim_name)
print(f'Saved: {out_data}')